In [1]:
import numpy as np
import scipy.stats as stats
import scipy.signal as signal
from jaxtyping import Float
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# muutils
from muutils.dbg import dbg_tensor
from muutils.jsonlines import jsonl_write, jsonl_load

# attention-motifs
from attention_motifs.bins import Bins
from attention_motifs.features import scalar_feature_table, prefix_dict
from attention_motifs.math import compute_envelope_params
from attention_motifs.transition_tensor import transition_tensor

In [2]:
def vec_features(
	arr: np.ndarray,
	compute_distribution: bool = True,
	compute_timeseries: bool = True,
) -> dict[str, float]:
	if len(arr.shape) != 1:
		dbg_tensor(arr)
		raise ValueError(f"Input arr must be 1-dimensional, got {arr.shape}")

	n: int = arr.size

	dist_features: dict[str, float] = dict()
	timeseries_features: dict[str, float] = dict()

	if compute_distribution:
		bins: int = 10 if n >= 10 else n
		hist, _ = np.histogram(arr, bins=bins)
		probs: np.ndarray = (
			hist.astype(float) / hist.sum() if hist.sum() > 0 else hist.astype(float)
		)
		dist_features = dict(
			mean=np.mean(arr),
			median=np.median(arr),
			variance=np.var(arr, ddof=1),
			std=np.std(arr, ddof=1),
			skewness=stats.skew(arr),
			kurtosis=stats.kurtosis(arr),
			entropy=stats.entropy(probs, base=2),
			L1_norm=np.sum(np.abs(arr)) / n,
			L2_norm=np.linalg.norm(arr, ord=2) / n,
			rms=np.sqrt(np.mean(arr**2)),
			energy=np.sum(arr**2),
		)

	if compute_timeseries:
		# Lag-1 Autocorrelation (Pearson correlation between arr[:-1] and arr[1:])
		autocorr_lag1: float = (
			np.corrcoef(arr[:-1], arr[1:])[0, 1]
			if np.std(arr[:-1]) > 0 and np.std(arr[1:]) > 0
			else 0.0
		)

		# PSD using Welch's method (total power)
		freqs, psd_vals = signal.welch(arr, nperseg=n)
		psd_total_power: float = np.sum(psd_vals)

		# Linear regression using scipy.stats.linregress
		t: np.ndarray = np.arange(n)
		linreg_result = stats.linregress(t, arr)
		line_fit: dict[str, float] = dict(
			slope=linreg_result.slope,
			intercept=linreg_result.intercept,
			r2=linreg_result.rvalue**2,
		)

		timeseries_features: dict[str, float] = dict(
			zero_crossing_rate=np.sum(np.diff(np.signbit(arr))) / (n - 1),
			autocorr_lag1=autocorr_lag1,
			psd_total_power=psd_total_power,
			**{f"linreg.{k}": v for k, v in line_fit.items()},
		)

	return {**dist_features, **timeseries_features}

In [16]:
def scaled_beta(x: np.ndarray, alpha: float, beta: float, scale: float) -> np.ndarray:
	"""Scaled beta distribution."""
	return stats.beta.pdf(x, alpha, beta) * scale


def hist_beta_fit(
	x: Float[np.ndarray, " k"],
	bins: Bins,
) -> dict[str, float]:
	x_hist, _ = np.histogram(x, bins.edges)
	popt, pcov = curve_fit(
		scaled_beta,
		bins.centers,
		x_hist,
		p0=(1.0, 1.0, 1.0),
	)

	return dict(
		alpha=popt[0],
		beta=popt[1],
		scale=popt[2],
		**prefix_dict(
			vec_features(x_hist - scaled_beta(bins.centers, *popt)),
			prefix="hist",
		),
	)

In [17]:
def skew_lt(
	L: Float[np.ndarray, "n n"],
) -> Float[np.ndarray, "n n"]:
	"""Shift rows of a lower-triangular matrix so that its diagonal becomes the rightmost column.

	# Parameters:
	 - `L : Float[np.ndarray, "n n"]`
	   A square lower-triangular matrix of shape `(n, n)`.

	# Returns:
	 - `Float[np.ndarray, "n n"]`
	   A matrix of shape `(n, n)` with rows shifted so the original diagonal is in the rightmost column.
	"""
	n: int = L.shape[0]
	S: Float[np.ndarray, "n n"] = np.zeros_like(L)
	i, j = np.tril_indices(n)  # row indices i, col indices j of the lower triangle
	S[i, j + (n - i - 1)] = L[i, j]  # shift columns so diagonal ends up at column n-1
	return S


def gram_features(A: Float[np.ndarray, "n_ctx n_ctx"]) -> dict[str, float]:
	dbg_tensor(A)
	return prefix_dict(
		hist_beta_fit(
			A.flatten(),
			bins=Bins(n_bins=32, start=0.0, stop=1.0),
		),
		prefix="beta_hist",
	)

In [18]:
def tt_features(
	A: Float[np.ndarray, "n_ctx n_ctx"],
	p_threshold: float = 0.0,
) -> dict[str, float]:
	idxs, tt, res = transition_tensor(A, exact=100, approx_l10=7.0, approx_pts=100)

	output: dict[str, float] = dict()

	indices_raw = np.apply_along_axis(
		lambda row: np.searchsorted(row, p_threshold, side="right"),
		axis=0,
		arr=tt[:, :, 0],
	)
	idxs_with_inf = np.concatenate((idxs, [1e10]))
	indices_adjusted = np.array(idxs_with_inf[indices_raw], dtype=float)
	# if last element, set to inf
	# indices_adjusted[indices_raw == len(idxs)] = 1e10
	indices_adjusted_l10 = np.log10(indices_adjusted[1:])
	# ax_tt_time.plot(indices_adjusted_l10, "ro")
	output.update(
		prefix_dict(
			vec_features(indices_adjusted_l10),
			prefix="time",
		)
	)
	idxs_x = np.arange(len(indices_adjusted_l10))
	for envtype in ("lower", "upper", "bestfit"):
		m, b, r2 = compute_envelope_params(
			x=idxs_x,
			y=indices_adjusted_l10,
			envelope_type=envtype,
		)
		output.update(
			prefix_dict(
				dict(
					slope=m,
					intercept=b,
					r2=r2,
				),
				prefix=["env", envtype],
			)
		)

	indices_adjusted_diff = np.diff(indices_adjusted)
	output.update(
		prefix_dict(
			vec_features(indices_adjusted_diff),
			prefix="diff",
		)
	)

	return output

In [19]:
def compute_scalar_features(
	A: Float[np.ndarray, "n_ctx n_ctx"],
) -> dict[str, float]:
	dbg_tensor(A)

	A_log: Float[np.ndarray, "n_ctx n_ctx"] = np.nan_to_num(np.log(A + 1e-9), nan=-10)
	dbg_tensor(A_log)

	A_skew: Float[np.ndarray, "n_ctx n_ctx"] = skew_lt(A_log)
	A_log_skew: Float[np.ndarray, "n_ctx n_ctx"] = skew_lt(A_log)

	return dict(
		# diagonal: standard features, fit diff to beta dist
		**prefix_dict(vec_features(A.diagonal()), prefix="diag"),
		# off-diagonal: standard features, fit diff to beta dist
		**prefix_dict(vec_features(A[:, 0]), prefix="first_tok"),
		# transition tensor: standard features, standard features on diff, linear envelope on transition time
		# 	TODO: standard features on decay rate
		**prefix_dict(
			tt_features(A),
			prefix="markov_transition",
		),
		# {log, raw} gram matrix of {rows, cols, rows of skewed}: beta fit hist
		# 	TODO: fit fft in `gram_features`, but this is expensive
		**prefix_dict(
			gram_features(A @ A.T),
			prefix=["gram", "row"],
		),
		**prefix_dict(
			gram_features(A.T @ A),
			prefix=["gram", "col"],
		),
		**prefix_dict(
			gram_features(A_skew.T @ A_skew),
			prefix=["gram", "skew"],
		),
		**prefix_dict(
			gram_features(A_log @ A_log.T),
			prefix=["log", "gram", "row"],
		),
		**prefix_dict(
			gram_features(A_log.T @ A_log),
			prefix=["log", "gram", "col"],
		),
		**prefix_dict(
			gram_features(A_log_skew.T @ A_log_skew),
			prefix=["log", "gram", "skew"],
		),
	)

In [20]:
df: pd.DataFrame = scalar_feature_table(features_func=compute_scalar_features)

models: ['pythia-14m', 'gemma-2b', 'gpt2-small', 'pythia-1b', 'gpt2-medium', 'tiny-stories-1M']
model: 'pythia-14m'
✔️  (0.00s) setting up paths                                                   
✔️  (0.01s) loading prompts                                                    
128 prompts loaded


prompts:   0%|          | 0/128 [00:00<?, ?it/s][ <ipykernel>:4 ] A: μ=0.01 σ=0.04 x̃=0.00 R=[0.00,1.00] ℙ=|█▃▂▂▂▁ | shape=(114,114) dtype=float32
[ <ipykernel>:7 ] A_log: μ=-13.37 σ=7.48 x̃=-13.78 R=[-20.72,0.00] ℙ=|█▂▄▅▇▇▅| shape=(114,114) dtype=float32
[ <ipykernel>:21 ] A: μ=0.03 σ=0.05 x̃=0.01 R=[0.00,1.00] ℙ=|█▄▃▂▁ ▁| shape=(114,114) dtype=float32
[ <ipykernel>:21 ] A: μ=0.01 σ=0.06 x̃=0.00 R=[0.00,3.91] ℙ=|█▁▁    | shape=(114,114) dtype=float32
prompts:   0%|          | 0/128 [00:03<?, ?it/s]


RuntimeError: Optimal parameters not found: Number of calls to function has reached maxfev = 800.

In [ ]:
path: str = "../data/scalar_features.jsonl.gz"
jsonl_write(path, df.to_dict(orient="records"), use_gzip=True)

In [ ]:
_temp_loaded = pd.DataFrame(jsonl_load(path))
df = _temp_loaded

In [ ]:
def plot_histograms_long(df: pd.DataFrame) -> None:
	"""Plot histograms for each feature with different models superimposed.

	This function assumes the DataFrame is in long format with columns:
	"model", "feat_name", "feat_val", and optionally "prompt", "layer", "head".

	# Parameters:
	 - `df : pd.DataFrame`
		 DataFrame containing the data.

	# Returns:
	 - `None`
		 Displays the histograms.
	"""
	features: list[str] = df["feat_name"].unique().tolist()
	models: list[str] = df["model"].unique().tolist()

	for feature in features:
		plt.figure()
		subset_feature: pd.DataFrame = df[df["feat_name"] == feature]
		for model in models:
			subset_model: pd.DataFrame = subset_feature[
				subset_feature["model"] == model
			]
			plt.hist(
				subset_model["feat_val"], bins=50, alpha=0.5, label=model, density=True
			)
		plt.xlabel(feature)
		plt.ylabel("Frequency")
		plt.title(f"Histogram of {feature} for different models")
		plt.legend()
		plt.show()


plot_histograms_long(df)